In [0]:
%run "../../utils/metrics_query"

In [0]:
%run "../../utils/cdc_validator"

In [0]:
%run "../../utils/audit_logger"

In [0]:
#NOTEBOOK 2 WITH MODULES 

from pyspark.sql.functions import col, lower
import time

# ===================================================================================
# CONFIGURATION
# ===================================================================================
dbutils.widgets.text("customer_id", "cust_001")
customer_id = dbutils.widgets.get("customer_id").lower()

SILVER_CATALOG = "hgi"
SILVER_SCHEMA  = "silver"
GOLD_CATALOG   = "hgi"
GOLD_SCHEMA    = "gold"

print("=" * 70)
print("03_Load_Silver_to_Gold_Augment_TestValidation")
print(f"  Customer : {customer_id}")
print("=" * 70)

# ===================================================================================
# MODULE 1 — METRICS QUERY (START)
# ===================================================================================
print("✅ MetricsQuery loaded")

# ===================================================================================
# MODULE 2 — CDC VALIDATOR (START)
# ===================================================================================
#%run ../../utils/cdc_validator.py
print("✅ CDC Validator loaded")

# ===================================================================================
# MODULE 3 — AUDIT LOGGER
# COMMENTED for dev/test environment
# Uncomment when running in production or client demo environment
# ===================================================================================
# %run ../../utils/audit_logger.py
# initialize_audit_tables()
# print("✅ AuditLogger loaded")

start_time = time.time()

# ===================================================================================
# STEP 4 — DATA QUALITY VALIDATION
# ===================================================================================
print("\n" + "="*70)
print("STEP 4 — DATA QUALITY VALIDATION")
print("="*70)

dq = {}

dq["persons_no_duplicate_emails"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM (
        SELECT mk_email, COUNT(*) AS c
        FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
        GROUP BY mk_email HAVING c > 1)
""").collect()[0]["cnt"] == 0

dq["persons_email_is_lowercase"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
    WHERE mk_email != lower(mk_email)
""").collect()[0]["cnt"] == 0

dq["persons_status_not_null"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons
    WHERE mk_status IS NULL
""").collect()[0]["cnt"] == 0

dq["companies_no_duplicate_domains"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM (
        SELECT mk_domain, COUNT(*) AS c
        FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
        GROUP BY mk_domain HAVING c > 1)
""").collect()[0]["cnt"] == 0

dq["companies_domain_is_lowercase"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
    WHERE mk_domain != lower(mk_domain)
""").collect()[0]["cnt"] == 0

dq["companies_status_not_null"] = spark.sql(f"""
    SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies
    WHERE mk_status IS NULL
""").collect()[0]["cnt"] == 0

total_persons   = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons").collect()[0]["cnt"]
ok_persons      = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons WHERE mk_status='ok'").collect()[0]["cnt"]
total_companies = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies").collect()[0]["cnt"]
ok_companies    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies WHERE mk_status='ok'").collect()[0]["cnt"]
total_contacts  = spark.sql(f"SELECT COUNT(*) AS cnt FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all").collect()[0]["cnt"]
contacts_w_co   = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations WHERE company__name IS NOT NULL").collect()[0]["cnt"]

dq["persons_enrichment_coverage_pct"]            = round((ok_persons / total_persons * 100) if total_persons > 0 else 0, 2)
dq["companies_enrichment_coverage_pct"]          = round((ok_companies / total_companies * 100) if total_companies > 0 else 0, 2)
dq["contacts_computations_company_coverage_pct"] = round((contacts_w_co / total_contacts * 100) if total_contacts > 0 else 0, 2)

priority1_total = spark.sql(f"""
    SELECT COUNT(DISTINCT lower(ca.email)) AS cnt
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
    JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
    WHERE ca.email IS NOT NULL
      AND uc.data_timestamp >= date_add(current_date(), -7)
""").collect()[0]["cnt"]

priority1_ok = spark.sql(f"""
    SELECT COUNT(DISTINCT lower(ca.email)) AS cnt
    FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
    JOIN {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts uc ON ca.id = uc.id
    JOIN {GOLD_CATALOG}.{GOLD_SCHEMA}.persons p ON lower(ca.email) = lower(p.mk_email)
    WHERE ca.email IS NOT NULL
      AND uc.data_timestamp >= date_add(current_date(), -7)
      AND p.mk_status = 'ok'
""").collect()[0]["cnt"]

priority1_rate = round((priority1_ok / priority1_total * 100) if priority1_total > 0 else 0, 2)
overall_rate   = dq["persons_enrichment_coverage_pct"]

dq["priority1_enrichment_rate_pct"]       = priority1_rate
dq["priority1_rate_exceeds_overall_rate"] = priority1_rate >= overall_rate

THRESHOLDS = {
    "persons_enrichment_coverage_pct":             60.0,
    "companies_enrichment_coverage_pct":           70.0,
    "contacts_computations_company_coverage_pct":  50.0,
    "priority1_enrichment_rate_pct":               overall_rate,
}

print("\n  DATA QUALITY REPORT")
print("  " + "-"*65)
all_passed = True
for check, result in dq.items():
    if isinstance(result, bool):
        status = "PASS" if result else "FAIL"
        if not result: all_passed = False
    else:
        threshold = THRESHOLDS.get(check, 0)
        flag = "OK" if result >= threshold else "BELOW THRESHOLD"
        status = f"{result}%  (min: {threshold}%)  {flag}"
        if result < threshold: all_passed = False
    print(f"  {check:<52}: {status}")
print("  " + "-"*65)
print("  All checks passed!" if all_passed else "  Some checks need review.")



# ===================================================================================
# MODULE 2 — CDC VALIDATOR (END)
# Verifies CDF is enabled on gold tables using cdc_validator module
# ===================================================================================
print("\n" + "="*70)
print("CDC / CDF VALIDATION — Verifying CDF on gold tables")
print("="*70)

cdf_result = {"status": False}
try:
    cdf_result = validate_cdc(
        layer       = "gold",
        table_names = ["persons", "companies", "contacts_computations"],
        catalog     = GOLD_CATALOG,
        schema      = GOLD_SCHEMA
    )
    print(f"\n  CDC Result: {cdf_result['message']}")
    print(f"  Enabled : {cdf_result['enabled_tables']}")
    print(f"  Disabled: {cdf_result['disabled_tables']}")

    if cdf_result["status"]:
        print("  CDF STATUS: ALL GOLD TABLES ENABLED")
    else:
        print("  CDF STATUS: WARNING — Some tables disabled")
        for tbl in cdf_result["disabled_tables"]:
            print(f"  Fix: spark.sql(\"ALTER TABLE {GOLD_CATALOG}.{GOLD_SCHEMA}.{tbl} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')\")")

except Exception as cdc_e:
    print(f"  WARNING: CDC validation failed: {str(cdc_e)}")

# ===================================================================================
# MODULE 1 — METRICS QUERY (END)
# ===================================================================================
print("\n" + "="*70)
print("METRICS — Saving pipeline stats")
print("="*70)

end_time    = time.time()
duration_ms = round((end_time - start_time) * 1000, 2)

try:
    persons_rows      = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.persons").collect()[0]["cnt"]
    companies_rows    = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.companies").collect()[0]["cnt"]
    computations_rows = spark.sql(f"SELECT COUNT(*) AS cnt FROM {GOLD_CATALOG}.{GOLD_SCHEMA}.contacts_computations").collect()[0]["cnt"]

    print(f"  hgi.gold.persons              : {persons_rows} rows")
    print(f"  hgi.gold.companies            : {companies_rows} rows")
    print(f"  hgi.gold.contacts_computations: {computations_rows} rows")

    total_rows = persons_rows + companies_rows + computations_rows
    mq.save_stats(days=30, rows_processed=total_rows)
    print(f"  Metrics saved | total rows_processed: {total_rows}")
except Exception as mq_e:
    print(f"  WARNING: Metrics save failed: {str(mq_e)}")

# ===================================================================================
# MODULE 3 — AUDIT LOGGER (END)
# ===================================================================================
# ==================================================================================
# PRODUCTION AUDIT LOGGING — Currently Commented for Dev/Test Environment
# ==================================================================================
# When you uncomment this section it will:
#   1. Write DQ results to hgi.gold.audit_logs table
#   2. Capture CDC validation status in the audit record
#   3. Send email alert (SUCCESS if all_passed, DQ_CHECK_FAILED if not)
#   4. Give full traceability of this test run in production audit trail
#
# HOW TO ACTIVATE:
#   1. Uncomment %run and initialize_audit_tables() at the top of this notebook
#   2. Uncomment log_audit() call below
#   3. Replace "your_email@company.com" with your actual email
#   4. Run through Databricks Jobs for full stats capture
# ==================================================================================
#
# log_audit(
#     job_name         = "03_Load_Silver_to_Gold_Augment_TestValidation",
#     customer_id      = customer_id,
#     status           = "success" if all_passed else "failure",
#     alert_type       = "SUCCESS" if all_passed else "DQ_CHECK_FAILED",
#     error_type       = None if all_passed else "DQ_THRESHOLD_NOT_MET",
#     error_reason     = None if all_passed else "One or more DQ thresholds not met",
#     layer            = "gold",
#     rows_processed   = total_rows,
#     duration_ms      = int(duration_ms),
#     email_on_failure = "pratibha.j.kumari@v4c.ai"
# )

print(f"""
=======================================================================
  TEST VALIDATION COMPLETE
  Customer : {customer_id}
  Status   : {"ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED"}
=======================================================================
  hgi.gold.persons              : {persons_rows} rows ({ok_persons} ok)
  hgi.gold.companies            : {companies_rows} rows ({ok_companies} ok)
  hgi.gold.contacts_computations: {computations_rows} rows
  Priority 1 rate               : {priority1_rate}% vs overall {overall_rate}%
  Duration                      : {duration_ms} ms
  CDF Status                    : {"ENABLED ON ALL" if cdf_result["status"] else "CHECK ABOVE"}
=======================================================================
""")

print("STEP 4 COMPLETE.")


  What this means:

persons_enrichment_coverage_pct: 41.4% (min: 60.0%) BELOW THRESHOLD

Out of 500 persons enriched, only 207 have mk_status = 'ok'. The rest are not_found. This is because only ~207 emails from contacts_all match person_profiles.

priority1_enrichment_rate_pct: 0.0% (min: 41.4%) FAIL



Priority 1 contacts (those joined with contacts table) — NONE of them got enriched with mk_status = 'ok'. All their emails returned not_found from person_profiles.

Root cause:
The contacts table and person_profiles have different emails. Contacts that join via Priority 1 path don’t exist in person_profiles.

How to fix:
Option 1 — Remove Priority 1 JOIN entirely:
Instead of joining contacts_all with contacts, just take ALL emails from contacts_all:

>> Priority 1 — take ALL contacts (no join with contacts table)
SELECT DISTINCT lower(ca.email) AS email, 1 AS priority
FROM {SILVER_CATALOG}.{SILVER_SCHEMA}.contacts_all ca
WHERE ca.email IS NOT NULL



Option 2 — Accept for POC:
Document as “faker data mismatch between contacts and person_profiles — will be resolved in production with real data.”
My recommendation: Option 2 for now. This is a faker data limitation, not a code issue. The logic is correct. In production with real Salesforce data, emails will match.



In [0]:
'''# Check if contacts_all has emails
spark.sql("SELECT COUNT(*) FROM hgi.silver.contacts_all WHERE email IS NOT NULL").show()

# Check data_timestamp range in contacts table
spark.sql("""
    SELECT MIN(data_timestamp), MAX(data_timestamp) 
    FROM hgi.silver.contacts
""").show()

# Check events date range
spark.sql("""
    SELECT MIN(event_timestamp), MAX(event_timestamp)
    FROM hgi.silver.events
""").show()'''

In [0]:
'''# Check if IDs match between contacts_all and contacts
spark.sql("""
    SELECT COUNT(*) FROM hgi.silver.contacts_all ca
    JOIN hgi.silver.contacts uc ON ca.id = uc.id
    WHERE ca.email IS NOT NULL
    AND uc.data_timestamp >= date_add(current_date(), -7)
""").show()

# Check contacts_all id vs contacts id format
spark.sql("SELECT id FROM hgi.silver.contacts_all LIMIT 3").show(truncate=False)
spark.sql("SELECT id FROM hgi.silver.contacts LIMIT 3").show(truncate=False)'''

In [0]:
'''spark.sql("""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT lower(ca.email) AS email
        FROM hgi.silver.contacts_all ca
        JOIN hgi.silver.contacts uc ON ca.id = uc.id
        WHERE ca.email IS NOT NULL
        AND uc.data_timestamp >= date_add(current_date(), -7)
    )
""").show()'''

In [0]:
'''spark.sql("""
    SELECT COUNT(*) FROM (
        WITH candidates AS (
            SELECT DISTINCT lower(ca.email) AS email, 1 AS priority
            FROM hgi.silver.contacts_all ca
            JOIN hgi.silver.contacts uc ON ca.id = uc.id
            WHERE ca.email IS NOT NULL
            AND uc.data_timestamp >= date_add(current_date(), -7)
        )
        SELECT r.email
        FROM candidates r
        LEFT JOIN hgi.gold.persons p ON r.email = lower(p.mk_email)
        WHERE p.mk_email IS NULL
           OR p.mk_status = 'error'
           OR p.mk_timestamp < date_add(current_date(), -180)
    )
""").show()'''

In [0]:
'''spark.sql("SELECT COUNT(*), COUNT(mk_email) FROM hgi.gold.persons").show()
spark.sql("SELECT mk_email, mk_status FROM hgi.gold.persons LIMIT 3").show(truncate=False)'''

In [0]:
'''# Check if enrichment tables exist and have data
spark.sql("SELECT COUNT(*) FROM hgi.enrichment.person_profiles").show()
spark.sql("SELECT * FROM hgi.enrichment.person_profiles LIMIT 3").show(truncate=False)'''

In [0]:
'''# Check if emails in contacts_all match emails in person_profiles
spark.sql("""
    SELECT COUNT(*) FROM hgi.silver.contacts_all ca
    JOIN hgi.enrichment.person_profiles p 
    ON lower(ca.email) = lower(p.mk_email)
    WHERE ca.email IS NOT NULL
""").show()

# Sample emails from both tables
spark.sql("SELECT lower(email) AS email FROM hgi.silver.contacts_all LIMIT 3").show(truncate=False)
spark.sql("SELECT lower(mk_email) AS mk_email FROM hgi.enrichment.person_profiles LIMIT 3").show(truncate=False)'''

In [0]:
'''spark.sql("""
    WITH candidates AS (
        SELECT DISTINCT lower(ca.email) AS email, 1 AS priority
        FROM hgi.silver.contacts_all ca
        JOIN hgi.silver.contacts uc ON ca.id = uc.id
        WHERE ca.email IS NOT NULL
        AND uc.data_timestamp >= date_add(current_date(), -7)
    ),
    ranked AS (
        SELECT email, MIN(priority) AS priority
        FROM candidates GROUP BY email
    )
    SELECT COUNT(*), 
           SUM(CASE WHEN p.mk_email IS NULL THEN 1 ELSE 0 END) AS null_matches,
           SUM(CASE WHEN p.mk_email IS NOT NULL THEN 1 ELSE 0 END) AS found_matches
    FROM ranked r
    LEFT JOIN hgi.gold.persons p ON r.email = lower(p.mk_email)
""").show()'''

In [0]:
'''# Check exact email format in both tables
spark.sql("SELECT email FROM hgi.silver.contacts_all LIMIT 5").show(truncate=False)
spark.sql("SELECT mk_email FROM hgi.enrichment.person_profiles LIMIT 5").show(truncate=False)

# Check if ANY emails match
spark.sql("""
    SELECT COUNT(*) FROM hgi.silver.contacts_all ca
    JOIN hgi.enrichment.person_profiles p
    ON lower(ca.email) = lower(p.mk_email)
""").show()'''

In [0]:
'''# Check how many of the 1000 matching emails are in the email candidates
spark.sql("""
    SELECT COUNT(*) FROM (
        SELECT lower(ca.email) AS email
        FROM hgi.silver.contacts_all ca
        JOIN hgi.enrichment.person_profiles p ON lower(ca.email) = lower(p.mk_email)
    ) matched
    JOIN (
        SELECT lower(ca.email) AS email
        FROM hgi.silver.contacts_all ca
        JOIN hgi.silver.contacts uc ON ca.id = uc.id
        WHERE uc.data_timestamp >= date_add(current_date(), -7)
    ) priority1
    ON matched.email = priority1.email
""").show()'''